# **Tokenization**
---

This notebook tokenizes the preprocessed and split .h5ad files using Geneformer's TranscriptomeTokenizer. The tokenization process converts gene expression profiles into a format that the Geneformer model can process.

## What is Tokenization?

Geneformer tokenizes single-cell transcriptomes by:
- Ranking genes by expression level within each cell
- Converting gene Ensembl IDs to token IDs using the model's vocabulary
- Creating token sequences that represent the transcriptional state of each cell

## Why Tokenize After Data Splitting?

**Critical workflow requirement:**
- The split labels (`l3_split` column) must be present in the h5ad file **before tokenization**
- Tokenization preserves metadata columns (including split labels) in the output
- The Geneformer `prepare_data` function uses these split labels to separate train/validation/test data

**Without split labels before tokenization:**
- Cannot split data properly for model training
- Would need to re-tokenize entire dataset

## Model Versions

**Geneformer V1:**
- Requires manual specification of gene dictionaries
- 30M parameter model trained on Genecorpus-30M

**Geneformer V2:**
- Automatically loads V2 token dictionary and gene mappings
- Improved vocabulary and tokenization

## Importance for Cell Type Classification Fine-Tuning

Proper tokenization ensures:
- Gene expression profiles are correctly encoded for the model
- Metadata (cell type labels, donor IDs, split labels) is preserved
- Training pipeline can correctly separate train/validation/test data
- Model receives biologically meaningful input sequences

---

## Setup Instructions

Before running this notebook, update the paths in the configuration cell below:
- `GENEFORMER_DIR`: Root directory for your Geneformer installation
- `BASE_DIR`: Root directory for the benchmarking repository
- `PREPROCESSED_SPLIT_DIR`: Directory containing preprocessed h5ad files with split labels (from notebook 02)
- `TOKENIZED_DIR`: Output directory for tokenized data
- `GENE_DICT_DIR`: Directory containing gene dictionaries (for V1 model)

For V2 model, gene dictionaries are loaded automatically.

In [4]:
import os
from pathlib import Path

# ===== CONFIGURATION - Update these paths for your environment =====
GENEFORMER_DIR = Path("/scratch/tmurugan/Geneformer")
BASE_DIR = Path("/scratch/tmurugan/geneformer_benchmarking")
DONOR_KEY = "p7"
PREPROCESSED_SPLIT_DIR = BASE_DIR / "data" / "preprocessed_data" / f"{DONOR_KEY}_test_donor_split"
TOKENIZED_DIR = BASE_DIR / "data" / "tokenized" / "citeseq_pbmc_tokenized"
GENE_DICT_DIR = GENEFORMER_DIR / "geneformer" / "gene_dictionaries_30m"

# Additional variables for tokenization
METADATA_COLS = ["donor_id", "joinid", "l3_split", "celltype.l1", "celltype.l3"]  # List of metadata columns to use for tokenization

# Create directories if they don't exist
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded:")
print(f"Geneformer directory: {GENEFORMER_DIR}")
print(f"Base directory: {BASE_DIR}")
print(f"Preprocessed directory: {PREPROCESSED_SPLIT_DIR}")
print(f"Tokenized output: {TOKENIZED_DIR}")

Configuration loaded:
Geneformer directory: /scratch/tmurugan/Geneformer
Base directory: /scratch/tmurugan/geneformer_benchmarking
Preprocessed directory: /scratch/tmurugan/geneformer_benchmarking/data/preprocessed_data/p7_test_donor_split
Tokenized output: /scratch/tmurugan/geneformer_benchmarking/data/tokenized/citeseq_pbmc_tokenized


## For Version 1
You need to specify the following files for Geneformer-V1-10M (which uses Genecorpus-30M for pretraining): 
  1. gene median dictionary,  
  2. token dictionary,  
  3. ensembl ID mapping dictionary. 

In [5]:
# Tokenization with Geneformer-V1 (TranscriptomeTokenizer)

from geneformer import TranscriptomeTokenizer
tokenizer = TranscriptomeTokenizer(
    custom_attr_name_dict={
        METADATA_COLS[0]:"donor_id", 
        METADATA_COLS[1]:"join_id", 
        METADATA_COLS[2]: "split",
        METADATA_COLS[3]:"cell_type_l1", 
        METADATA_COLS[4]:"cell_type_l3",
        
    },
    model_input_size=2048,
    special_token=False,
    model_version="V1",
    gene_median_file=str(GENE_DICT_DIR / "gene_median_dictionary_gc30M.pkl"),
    token_dictionary_file=str(GENE_DICT_DIR / "token_dictionary_gc30M.pkl"),
    gene_mapping_file=str(GENE_DICT_DIR / "ensembl_mapping_dict_gc30M.pkl")
)

tokenizer.tokenize_data(
    data_directory=str(PREPROCESSED_SPLIT_DIR),
    output_directory=str(TOKENIZED_DIR),
    output_prefix="v1_citeseq_pbmc_l3_tokenized",
    file_format="h5ad"
)
print("Tokenization finished; check folder")

Tokenizing /scratch/tmurugan/geneformer_benchmarking/data/preprocessed_data/p7_test_donor_split/citeseq_pbmc_with_p7_l3_split.h5ad


/scratch/tmurugan/conda_envs/env_geneformer/lib/python3.11/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/scratch/tmurugan/conda_envs/env_geneformer/lib/python3.11/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/scratch/tmurugan/geneformer_benchmarking/data/preprocessed_data/p7_test_donor_split/citeseq_pbmc_with_p7_l3_split.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
Tokenization finished; check folder


## For Version 2


In [6]:
# Tokenization with Geneformer-V2 (TranscriptomeTokenizer)
# Tokenizing celltype.l3 labels with Geneformer V2 token dictionary and mapping files

from geneformer import TranscriptomeTokenizer
# the default model_version="V2" will automatically load the V2 token dictionary and gene mapping files, so we don't need to specify those here
tokenizer = TranscriptomeTokenizer(
    custom_attr_name_dict={
        METADATA_COLS[0]:"donor_id", 
        METADATA_COLS[1]:"join_id", 
        METADATA_COLS[2]: "split",
        METADATA_COLS[3]:"cell_type_l1", 
        METADATA_COLS[4]:"cell_type_l3",
    }
)

tokenizer.tokenize_data(
    data_directory=str(PREPROCESSED_SPLIT_DIR),
    output_directory=str(TOKENIZED_DIR),
    output_prefix="v2_citeseq_pbmc_l3_tokenized",
    file_format="h5ad"
)
print("Tokenization finished; check folder")

Tokenizing /scratch/tmurugan/geneformer_benchmarking/data/preprocessed_data/p7_test_donor_split/citeseq_pbmc_with_p7_l3_split.h5ad


100%|██████████| 316/316 [01:39<00:00,  3.17it/s]
/scratch/tmurugan/conda_envs/env_geneformer/lib/python3.11/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/scratch/tmurugan/conda_envs/env_geneformer/lib/python3.11/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/scratch/tmurugan/geneformer_benchmarking/data/preprocessed_data/p7_test_donor_split/citeseq_pbmc_with_p7_l3_split.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
Tokenization finished; check folder
